# 🚀 ReqGPT / ThinkingPods — 100% Free Cloud Runner

Run ThinkingPods completely in the cloud on **Google Cloud's free 12.7 GB RAM & GPU infrastructure** without using your local computer's resources.

### ⚡ Quick Start:
1. Click **Runtime ➡️ Run all** in the top menu.
2. Scroll to the bottom cell to see your generated Cloudflare backend URL.
3. Open the permanent web app at **[ThinkingPods Live Web App](https://bezaleelpaul.github.io/ThinkingPods-Live/)** and paste the backend URL to start chatting!

In [ ]:
# Step 1: Clone the repository and install system dependencies
!apt-get update -qq && apt-get install -y -qq ffmpeg curl libportaudio2
!git clone https://github.com/BezaleelPaul/ThinkingPods-Live.git ThinkingPods
%cd ThinkingPods
!pip install -q -r requirements.txt
print("✅ Codebase and Python dependencies installed!")

In [ ]:
# Step 2: Install and Start Ollama with llama3.2:1b
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &

import time, urllib.request
print("Waiting for Ollama service to start...")
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags")
        break
    except Exception:
        time.sleep(1)

!ollama pull llama3.2:1b
print("✅ Ollama and llama3.2:1b ready on Google Cloud!")

In [ ]:
# Step 3: Install Cloudflare Tunnel for Free Public URL
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print("✅ Cloudflared tunnel ready!")

In [ ]:
# Step 4: Start Backend and Cloudflare Tunnel
import subprocess, time, re, urllib.request

# Start FastAPI server
print("Starting FastAPI backend...")
backend = subprocess.Popen(["python", "server.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for backend to be ready
for _ in range(35):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health")
        break
    except Exception:
        time.sleep(1)
print("✅ Backend is live on port 8000!")

# Start Cloudflare Tunnel for Backend API (to connect with GitHub Pages)
print("Generating free public cloud URL...")
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

backend_url = None
for line in tunnel.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        backend_url = match.group(0)
        break

print("\n" + "="*65)
print("🎉 ThinkingPods Backend is LIVE on Google Cloud!")
print(f"🔗 Cloud Backend API URL: {backend_url}")
print(f"👉 Open your Web App at: https://bezaleelpaul.github.io/ThinkingPods-Live/")
print("="*65 + "\n")

# Keep cell alive to keep services running
try:
    tunnel.wait()
except KeyboardInterrupt:
    print("Shutting down...")
    tunnel.terminate()
    backend.terminate()